In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# CSV 파일 경로
csv_path = "food_nutrition_data.csv"

# 데이터 불러오기
food_data = pd.read_csv(csv_path)

# NaN 값 처리 (0으로 채우기)
food_data = food_data.fillna(0)

# 데이터 확인
food_data.head()


,식품명(한글),칼로리,탄수화물,단백질,지방,1회 섭취량
0,가리비,2358.0,113.97,197.08,119.08,1086g
1,가지,24.0,5.70,1.01,0.19,100g
2,가지구이,220.0,3.00,14.00,17.00,1 link
3,가츠동,307.0,4.45,12.65,26.10,100g
4,간장,53.0,7.61,6.28,0.04,100g


In [3]:
# '1회 섭취량'에서 숫자만 추출하여 정수형으로 변환
food_data["1회 섭취량"] = food_data["1회 섭취량"].astype(str).str.extract("(\d+)")
food_data["1회 섭취량"] = pd.to_numeric(food_data["1회 섭취량"], errors='coerce').fillna(0)

# 데이터 타입 변환 (칼로리, 탄수화물, 단백질, 지방)
cols = ["칼로리", "탄수화물", "단백질", "지방", "1회 섭취량"]
food_data[cols] = food_data[cols].astype(float)

# 정리된 데이터 확인
food_data.head()


,식품명(한글),칼로리,탄수화물,단백질,지방,1회 섭취량
0,가리비,2358.0,113.97,197.08,119.08,1086.0
1,가지,24.0,5.70,1.01,0.19,100.0
2,가지구이,220.0,3.00,14.00,17.00,1.0
3,가츠동,307.0,4.45,12.65,26.10,100.0
4,간장,53.0,7.61,6.28,0.04,100.0


In [4]:
import numpy as np

def recommend_meal(target_calories, carb_ratio, protein_ratio, fat_ratio):
    """ 목표 칼로리에 맞춰 탄수화물, 단백질, 지방 비율에 맞는 식단 추천 """
    selected_foods = []
    remaining_calories = target_calories

    while remaining_calories > 50:  # 50kcal 이하일 때 종료
        # 무작위 음식 선택
        food = food_data.sample(n=1).iloc[0]

        food_calories = food["칼로리"]
        food_carbs = food["탄수화물"]
        food_protein = food["단백질"]
        food_fat = food["지방"]
        serving_size = food["1회 섭취량"]

        # 섭취량 비율 조정
        scale = min(1, remaining_calories / food_calories)  # 비율 조정
        selected_foods.append({
            "식품명": food["식품명(한글)"],
            "섭취량": round(serving_size * scale, 1),  # g 단위 조정
            "칼로리": round(food_calories * scale, 1),
            "탄수화물": round(food_carbs * scale, 1),
            "단백질": round(food_protein * scale, 1),
            "지방": round(food_fat * scale, 1),
        })

        remaining_calories -= food_calories * scale

    return selected_foods

# 예제 입력 (예측된 칼로리 값)
total_calories = 2000  # 예측된 총 섭취 칼로리
carbs = total_calories * 0.5 / 4  # 50% 탄수화물
protein = total_calories * 0.3 / 4  # 30% 단백질
fat = total_calories * 0.2 / 9  # 20% 지방

# 아침, 점심, 저녁 식단 추천
breakfast = recommend_meal(total_calories * 0.3, 0.5, 0.3, 0.2)  # 아침 (균형)
lunch = recommend_meal(total_calories * 0.4, 0.4, 0.4, 0.2)  # 점심 (고단백)
dinner = recommend_meal(total_calories * 0.3, 0.3, 0.4, 0.3)  # 저녁 (저탄수)

# 추천된 식단 확인
{"아침": breakfast, "점심": lunch, "저녁": dinner}


C:\Users\fn45\AppData\Local\Temp\ipykernel_9764\3417497822.py:19: RuntimeWarning: divide by zero encountered in scalar divide
  scale = min(1, remaining_calories / food_calories)  # 비율 조정


{'아침': [{'식품명': '가지구이',
   '섭취량': 1.0,
   '칼로리': 220.0,
   '탄수화물': 3.0,
   '단백질': 14.0,
   '지방': 17.0},
  {'식품명': '쇠고기구이',
   '섭취량': 101.0,
   '칼로리': 254.0,
   '탄수화물': 0.0,
   '단백질': 27.5,
   '지방': 15.1},
  {'식품명': '간장', '섭취량': 100.0, '칼로리': 53.0, '탄수화물': 7.6, '단백질': 6.3, '지방': 0.0},
  {'식품명': '돼지감자', '섭취량': 0.0, '칼로리': 0.0, '탄수화물': 0.0, '단백질': 0.0, '지방': 0.0},
  {'식품명': '오븐구이치킨몸통',
   '섭취량': 24.5,
   '칼로리': 73.0,
   '탄수화물': 0.0,
   '단백질': 6.3,
   '지방': 5.1}],
 '점심': [{'식품명': '까르보나라',
   '섭취량': 419.0,
   '칼로리': 800.0,
   '탄수화물': 107.7,
   '단백질': 33.7,
   '지방': 22.2}],
 '저녁': [{'식품명': '폭립',
   '섭취량': 0.0,
   '칼로리': 0.0,
   '탄수화물': 0.0,
   '단백질': 0.0,
   '지방': 0.0},
  {'식품명': '봉골레파스타',
   '섭취량': 100.0,
   '칼로리': 264.0,
   '탄수화물': 38.0,
   '단백질': 2.4,
   '지방': 12.5},
  {'식품명': '대파', '섭취량': 0.0, '칼로리': 0.0, '탄수화물': 0.0, '단백질': 0.0, '지방': 0.0},
  {'식품명': '닭가슴살시저샐러드',
   '섭취량': 101.0,
   '칼로리': 197.0,
   '탄수화물': 0.0,
   '단백질': 29.8,
   '지방': 7.8},
  {'식품명': '프레즐',
   '섭취량': 2.0,
   '칼로리': 13

In [5]:
# 1️⃣ CSV 파일 불러오기
file_path = "food_nutrition_data.csv"  # 파일 경로
food_data = pd.read_csv(file_path)

# 2️⃣ NaN 값 처리 및 데이터 전처리
food_data.fillna(0, inplace=True)  # NaN 값을 0으로 대체

# 3️⃣ 필요 컬럼 선택
X = food_data[['칼로리', '탄수화물', '단백질', '지방']].values  # 입력 변수
Y = food_data[['칼로리']].values  # 목표 칼로리 (예측 대상)

# 4️⃣ 데이터 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 5️⃣ 데이터 분할 (훈련 80% / 테스트 20%)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)

# 6️⃣ XGBoost 모델 학습
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=6)
model.fit(X_train, y_train)

# 7️⃣ 모델 저장
joblib.dump(model, "xgboost_meal_recommendation.pkl")

print("✅ XGBoost 식단 추천 모델 저장 완료!")


✅ XGBoost 식단 추천 모델 저장 완료!
